# >>> MODIFICATION LOG: shared 12-output head -> 12 independent per-task heads

This notebook is the existing OrthoFoundation DINOv3 + hierarchical MIL pipeline,
**modified in place**. The data-loading pipeline, DINO embedding loader, Window/Series
Attention MIL, train/validation split, optimizer/LR config, and early-stopping
methodology are all **unchanged**. Only the classification head and the
config/checkpoint/evaluation plumbing around it were touched.

**Changed cells** (search for `# [MULTI-HEAD MOD]` comments inline):
- **Config cell**: new `MULTIHEAD_HIDDEN_DIMS` / `MULTIHEAD_DROPOUT` settings; all
  checkpoint/output paths redirected to a **new, separate output directory** so the
  original shared-head run's checkpoints are never touched or overwritten.
- **Model cell**: the single shared `Linear(512,128)->ReLU->Linear(128,12)` head is
  replaced with `MultiTaskHeads`, a `nn.ModuleList` of 12 independent
  `512 -> 128 -> 64 -> 1` MLPs (ReLU + dropout), concatenated into one
  `(batch_size, 12)` logits tensor. `OrthoDinoMILModel.forward()` is otherwise
  identical — projection, Window MIL, and Series MIL are untouched.
- **Stage 1 init cell**: loading a Stage 1 head into the new per-task heads is no
  longer shape-compatible, so that one load is explicitly skipped (with a clear
  printed explanation) when `INIT_FROM_STAGE1=True`; Window MIL / Series MIL
  loading is unaffected.
- **Batch construction cell**: added `.pin_memory()` on the collated tensors (GPU
  transfer efficiency) — the padding/masking logic itself is unchanged.
- **Checkpoint cell**: `torch.load(..., weights_only=False)` added explicitly on every
  load of a file this notebook itself generates, for PyTorch 2.6+ compatibility.
- **Final verification / summary cells**: added a best-effort comparison against the
  old shared-head model's saved results, if that file exists.

Everything else below is the original notebook, unmodified.


# OrthoFoundation DINOv3 Embeddings -> Hierarchical MIL

Trains Window MIL -> Series MIL -> classification head directly on **precomputed,
frozen OrthoFoundation DINOv3 embeddings**. No DICOMs, no ResNet, no new feature
extraction. A trainable `Linear(DINO_DIM, 512)` projection (+ activation + norm)
adapts the DINOv3 dimensionality to the 512-D input the MIL stack expects.

**A note on the embedding file inspection this task asked for:** I was not able to
browse or download the actual contents of the
`prithvirajrp/orthofoundation-rsna-knee-raw-embeddings` Kaggle dataset from this
environment (Kaggle dataset pages render client-side and I have no Kaggle
credentials/API access here) — so I could not confirm the exact file format/keys
ahead of time the way I did for the in-chat-uploaded notebooks in earlier stages.
To compensate, **Section 4 below inspects the real files live, first thing, when you
run this on Kaggle**, printing extensions found, raw keys, nested types and shapes.
Section 5's loader is written to recognize several plausible layouts (nested
`dict[study][series]->array`, a flat `dict[study]->array`, or a single-record `dict`
with explicit `StudyInstanceUID`/`embeddings`/`series_ids`/`window_order` keys) and
is isolated in one function (`parse_container`) so it's easy to adjust if the printed
structure doesn't match any of those cases.


## 1. Imports

In [25]:
import os
import re
import json
import glob
import random
import pickle
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm


## 2. Config

In [26]:
CONFIG = {
    # Data
    "EMBEDDINGS_DIR": "/kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings",
    "LABELS_CSV": "/kaggle/input/datasets/flight0234/rsna-knee-hybrid-report-labels/report_labels_v4hybrid.csv",
    "SAMPLE_CSV": "/kaggle/input/datasets/prithvirajrp/rsna-train-distribution-subset/resnet_finetune_1000.csv",
    # [GOLD-VAL MOD] New: the ORIGINAL competition train.csv, used only to pull
    # out the 58 studies that have complete official binary labels ("gold").
    # This path follows the same /kaggle/input/competitions/<slug>/... pattern
    # used elsewhere for this competition's data -- VERIFY this is the actual
    # mount path in your environment before running; I could not browse Kaggle
    # to confirm it.
    "COMPETITION_TRAIN_CSV": "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv",
    "ID_COL": "StudyInstanceUID",
    "NUM_CLASSES": 12,
    "PROJECTED_DIM": 512,

    # Deterministic 850/150 split (same recipe/seed as before -- UNCHANGED)
    "SPLIT_SEED": 42,
    "N_TRAIN": 850,

    # Full-data split
    
    "TRAIN_FRACTION": 0.80,

    # Training (UNCHANGED: same optimizer/LR/batch size/early-stopping as the
    # original shared-head run, per the request to preserve these)
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 25,
    "EARLY_STOPPING_PATIENCE": 3,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "MAX_GRAD_NORM": 1.0,
    "SHUFFLE_SEED": 1337,
    "L1_LAMBDA" :5e-6,

    # Shared MIL dims (UNCHANGED -- same feature extractor/aggregation as before)
    "ATTENTION_HIDDEN_DIM": 128,

    # [MULTI-HEAD MOD] Old shared-head config (kept only for reference / loading the
    # old model for comparison -- no longer used to build the new model).
    "HEAD_HIDDEN_DIM": 128,  # was: shared 512 -> 128 -> 12 head

    # [ARCHITECTURE MOD] No longer used by the model: the classifier is now a
    # fixed single Linear(512,1) per task (no hidden layers, no dropout). Kept
    # here only so old checkpoints' stored CONFIG remains self-describing.
    "MULTIHEAD_HIDDEN_DIMS": (128, 64),
    "MULTIHEAD_DROPOUT": 0.2,

    # Stage 1 weight initialization (Option A vs Option B) -- UNCHANGED mechanism,
    # but see Section 10: the head can no longer be loaded from Stage 1 because the
    # architecture changed shape (shared 12-way head -> 12 independent heads).
    "INIT_FROM_STAGE1": False,
    "STAGE1_SEARCH_DIRS": ["/kaggle/working/STAGE1_OUTPUTS", "/kaggle/input", "/kaggle/working"],
    "STAGE1_WINDOW_MIL_FILENAME": "STAGE1_window_mil.pt",
    "STAGE1_SERIES_MIL_FILENAME": "STAGE1_series_mil.pt",
    "STAGE1_HEAD_FILENAME": "STAGE1_head.pt",

    # Checkpointing (UNCHANGED cadence/contents policy)
    "CHECKPOINT_EVERY_N_STUDIES": 100,

    # [MULTI-HEAD MOD] Old (shared-head) output directory -- READ-ONLY from here on,
    # used only to load old results for the comparison step in Section 19. Nothing
    # in this notebook writes to this path anymore.
    "OLD_OUTPUT_DIR": "/kaggle/working/ORTHOFOUNDATION_DINO_MIL_OUTPUTS",

    # [MULTI-HEAD MOD] Separate output directory from the original shared-head run.
    # [GOLD-VAL MOD] Pointed at a NEW, further-separate directory for this
    # experiment specifically, since the validation methodology (58-study gold
    # macro ROC-AUC instead of a random 80/20 split) is incompatible with
    # checkpoints from the previous split -- this prevents accidentally
    # resuming into, or overwriting, those results.
    "OUTPUT_DIR": "/kaggle/working/ORTHOFOUNDATION_DINO_MULTIHEAD_GOLDVAL_OUTPUTS",
}

h1 = 512
h2 = 128

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

# [MULTI-HEAD MOD] All checkpoint/output filenames below point into the NEW
# directory and carry a MULTIHEAD prefix, so they can never collide with the
# original shared-head run's files.
LATEST_CKPT_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_latest_checkpoint.pt")
BEST_CKPT_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_best_checkpoint.pt")
FINAL_MODEL_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_final_model.pt")
HISTORY_CSV_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_training_history.csv")
CONFIG_JSON_PATH = os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_config.json")

# [MULTI-HEAD MOD] Old final-model path, read-only, used only for the Section 19
# comparison (loaded defensively -- the run proceeds fine if this doesn't exist).
OLD_FINAL_MODEL_PATH = os.path.join(CONFIG["OLD_OUTPUT_DIR"], "ORTHO_DINO_MIL_final_model.pt")

with open(CONFIG_JSON_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("NEW (multi-head) output dir:", CONFIG["OUTPUT_DIR"])
print("OLD (shared-head) output dir (read-only, for comparison only):", CONFIG["OLD_OUTPUT_DIR"])
print(
    "\nIMPORTANT: /kaggle/working is temporary storage tied to this session. "
    "Nothing here is permanently stored. Before the session ends, either download "
    f"{CONFIG['OUTPUT_DIR']} or save this notebook's run as a Kaggle 'Output' "
    "(Save Version -> Save & Run All) so the files persist as a notebook output."
)


NEW (multi-head) output dir: /kaggle/working/ORTHOFOUNDATION_DINO_MULTIHEAD_GOLDVAL_OUTPUTS
OLD (shared-head) output dir (read-only, for comparison only): /kaggle/working/ORTHOFOUNDATION_DINO_MIL_OUTPUTS

IMPORTANT: /kaggle/working is temporary storage tied to this session. Nothing here is permanently stored. Before the session ends, either download /kaggle/working/ORTHOFOUNDATION_DINO_MULTIHEAD_GOLDVAL_OUTPUTS or save this notebook's run as a Kaggle 'Output' (Save Version -> Save & Run All) so the files persist as a notebook output.


## 3. Device / AMP setup

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_TYPE = device.type
USE_AMP = torch.cuda.is_available()

print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("AMP enabled:", USE_AMP)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"Total GPU memory: {props.total_memory / (1024**3):.2f} GB")
    print(f"Free/allocated at startup: "
          f"{torch.cuda.memory_reserved() / (1024**3):.2f} GB reserved, "
          f"{torch.cuda.memory_allocated() / (1024**3):.2f} GB allocated")


CUDA available: True
Device: cuda
AMP enabled: True
GPU: Tesla T4
Total GPU memory: 14.56 GB
Free/allocated at startup: 2.13 GB reserved, 0.31 GB allocated


## 4. Inspect the actual embedding files (UNCHANGED)

Finds every file under `EMBEDDINGS_DIR` regardless of extension, groups by extension,
and prints the raw structure of a sample file per extension.


In [28]:
def find_all_files(root):
    all_files = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            all_files.append(os.path.join(dirpath, fn))
    return all_files


all_embedding_files = find_all_files(CONFIG["EMBEDDINGS_DIR"])
print(f"Total files found under {CONFIG['EMBEDDINGS_DIR']}: {len(all_embedding_files)}")

by_ext = {}
for p in all_embedding_files:
    ext = os.path.splitext(p)[1].lower()
    by_ext.setdefault(ext, []).append(p)

print("\nFiles by extension:")
for ext, paths in sorted(by_ext.items(), key=lambda kv: -len(kv[1])):
    print(f"  {ext or '(no extension)'}: {len(paths)} file(s)  e.g. {paths[0]}")

if not all_embedding_files:
    raise FileNotFoundError(
        f"No files found under {CONFIG['EMBEDDINGS_DIR']!r}. Update CONFIG['EMBEDDINGS_DIR'] "
        "to the actual mount path for this dataset in your Kaggle environment."
    )


Total files found under /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings: 18

Files by extension:
  .npy 11: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 11
  .npy 12: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 12
  .npy 14: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 14
  .npy: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy
  .npy 5: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 5
  .npy 3: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 3
  .npy 6: 1 file(s)  e.g. /kaggle/input/datasets/prithvirajrp/orthofoundation-rsna-knee-raw-embeddings/features.npy 6
  .npy 10: 1 file(s)  e.g. /kaggle/input/datas

In [29]:

def _to_str_key(x):
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore")
    return str(x)


def _as_window_array(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()

    if isinstance(x, dict):
        raise ValueError(
            "Expected a feature array but found a nested dictionary."
        )

    arr = np.asarray(x, dtype=np.float32)

    if arr.ndim == 1:
        arr = arr[None, :]

    if arr.ndim != 2:
        raise ValueError(
            f"Expected [num_windows, embedding_dim], got {arr.shape}"
        )

    if arr.shape[0] == 0:
        return None

    return arr


def load_embeddings_index(files):
    merged = {}

    for file_idx, path in enumerate(files, 1):
        print(f"\n[{file_idx}/{len(files)}] Loading {os.path.basename(path)}")

        # Load .npy directly. This also handles names like features.npy 11,
        # which os.path.splitext() does not identify as having extension .npy.
        raw = np.load(path, allow_pickle=True)
        obj = raw.item() if raw.shape == () else raw

        if not isinstance(obj, dict):
            raise ValueError(
                f"Expected a dictionary in {path}, got {type(obj)}"
            )

        file_studies = 0

        for study_key, study_value in obj.items():
            study_id = _to_str_key(study_key)

            if not isinstance(study_value, dict):
                raise ValueError(
                    f"Expected series dictionary for study {study_id}, "
                    f"got {type(study_value)}"
                )

            if study_id not in merged:
                merged[study_id] = {}

            for series_key, feature_value in study_value.items():
                series_id = _to_str_key(series_key)
                arr = _as_window_array(feature_value)

                if arr is not None:
                    merged[study_id][series_id] = arr

            file_studies += 1

        print(f"  Studies loaded: {file_studies}")

    print("\n" + "=" * 60)
    print("EMBEDDING MERGE COMPLETE")
    print("=" * 60)
    print("Files merged:", len(files))
    print("Total studies:", len(merged))

    return merged


embeddings_index = load_embeddings_index(all_embedding_files)

# Verify actual embedding dimensions and grouping
sample_dims = set()
series_counts = []
window_counts = []

for study_id, series_dict in list(embeddings_index.items())[:100]:
    series_counts.append(len(series_dict))

    for arr in series_dict.values():
        sample_dims.add(arr.shape[-1])
        window_counts.append(arr.shape[0])

print("Embedding dimensions:", sample_dims)

if len(sample_dims) != 1:
    raise ValueError(f"Inconsistent or missing embedding dimensions: {sample_dims}")

DINO_DIM = next(iter(sample_dims))
print("DINO_DIM:", DINO_DIM)

if series_counts:
    print("Series/study — min:", min(series_counts),
          "max:", max(series_counts),
          "mean:", np.mean(series_counts))

if window_counts:
    print("Windows/series — min:", min(window_counts),
          "max:", max(window_counts),
          "mean:", np.mean(window_counts))



[1/18] Loading features.npy 11
  Studies loaded: 250

[2/18] Loading features.npy 12
  Studies loaded: 250

[3/18] Loading features.npy 14
  Studies loaded: 250

[4/18] Loading features.npy
  Studies loaded: 250

[5/18] Loading features.npy 5
  Studies loaded: 250

[6/18] Loading features.npy 3
  Studies loaded: 250

[7/18] Loading features.npy 6
  Studies loaded: 250

[8/18] Loading features.npy 10
  Studies loaded: 250

[9/18] Loading features.npy 7
  Studies loaded: 250

[10/18] Loading features.npy 16
  Studies loaded: 250

[11/18] Loading features.npy 13
  Studies loaded: 250

[12/18] Loading features.npy 15
  Studies loaded: 250

[13/18] Loading features.npy 4
  Studies loaded: 250

[14/18] Loading features.npy 2
  Studies loaded: 250

[15/18] Loading features.npy 9
  Studies loaded: 250

[16/18] Loading features.npy 18
  Studies loaded: 250

[17/18] Loading features.npy 17
  Studies loaded: 157

[18/18] Loading features.npy 8
  Studies loaded: 250

EMBEDDING MERGE COMPLETE
File

## 5. Generic, adaptable embeddings loader (UNCHANGED)

Recognizes a nested `dict[study_id -> dict[series_id -> array]]`, a flat
`dict[study_id -> array]`, or a single-record `dict` with explicit
`StudyInstanceUID`/`embeddings`/`series_ids`/`window_order` keys.


In [30]:
# def _to_str_key(k):
#     if isinstance(k, bytes):
#         return k.decode("utf-8", errors="ignore")
#     return str(k)


# def _as_window_array(v):
#     if v is None:
#         return None
#     if torch.is_tensor(v):
#         v = v.detach().cpu().numpy()
#     arr = np.asarray(v, dtype=np.float32)
#     if arr.ndim == 1:
#         arr = arr[None, :]
#     return arr


# def _get_first_present(d, candidates):
#     for c in candidates:
#         if c in d:
#             return d[c]
#     return None


# def _split_by_series(emb_val, series_ids, order_val):
#     arr = _as_window_array(emb_val)
#     if arr is None or arr.shape[0] == 0:
#         return {}

#     if order_val is not None:
#         order_idx = np.argsort(np.asarray(order_val).astype(np.int64))
#         arr = arr[order_idx]
#         if series_ids is not None:
#             series_ids = np.asarray(list(series_ids))[order_idx]

#     if series_ids is None:
#         return {"series_0": arr}

#     series_ids = np.asarray([_to_str_key(s) for s in series_ids])
#     series_dict = {}
#     for sid in np.unique(series_ids):
#         series_dict[sid] = arr[series_ids == sid]
#     return series_dict


# def parse_container(obj, fallback_study_id=None):
#     """Returns dict[study_id -> dict[series_id -> np.ndarray[num_windows, DINO_DIM]]]."""
#     result = {}

#     if isinstance(obj, dict):
#         id_val = _get_first_present(obj, ["StudyInstanceUID", "study_id", "studyinstanceuid", "study"])
#         emb_val = _get_first_present(obj, ["embeddings", "features", "embedding", "feature"])

#         if id_val is not None and emb_val is not None:
#             study_id = _to_str_key(id_val)
#             series_ids = _get_first_present(obj, ["series_ids", "SeriesInstanceUID", "series", "series_id"])
#             order_val = _get_first_present(obj, ["window_order", "order", "slice_index", "window_index"])
#             result[study_id] = _split_by_series(emb_val, series_ids, order_val)
#             return result

#         for k, v in obj.items():
#             study_id = _to_str_key(k)
#             if isinstance(v, dict):
#                 series_dict = {}
#                 for sk, sv in v.items():
#                     arr = _as_window_array(sv)
#                     if arr is not None and arr.shape[0] > 0:
#                         series_dict[_to_str_key(sk)] = arr
#                 result[study_id] = series_dict
#             else:
#                 arr = _as_window_array(v)
#                 if arr is not None and arr.shape[0] > 0:
#                     result[study_id] = {"series_0": arr}
#         return result

#     if fallback_study_id is not None:
#         arr = _as_window_array(obj)
#         if arr is not None and arr.shape[0] > 0:
#             return {fallback_study_id: {"series_0": arr}}

#     return result


# def load_embeddings_index(files):
#     merged = {}
#     n_parsed = 0

#     for path in files:
#         ext = os.path.splitext(path)[1].lower()
#         if ext not in (".npz", ".pt", ".pth", ".pkl", ".npy"):
#             continue
#         try:
#             obj, _ = load_raw(path)
#         except Exception as e:
#             print(f"WARNING: could not load {path}: {e}")
#             continue

#         fallback_id = os.path.splitext(os.path.basename(path))[0]
#         parsed = parse_container(obj, fallback_study_id=fallback_id)

#         for study_id, series_dict in parsed.items():
#             if study_id in merged:
#                 merged[study_id].update(series_dict)
#             else:
#                 merged[study_id] = series_dict

#         n_parsed += 1

#     print(f"Parsed {n_parsed}/{len(files)} embedding file(s).")
#     print(f"Total studies found in embeddings index: {len(merged)}")
#     return merged


# embeddings_index = load_embeddings_index(all_embedding_files)

# sample_dims = set()
# for series_dict in list(embeddings_index.values())[:50]:
#     for arr in series_dict.values():
#         sample_dims.add(arr.shape[-1])

# print("Observed DINOv3 embedding dimensionality (sampled):", sample_dims)
# if len(sample_dims) != 1:
#     print("WARNING: inconsistent or empty embedding dimensionality found. Inspect Section 4's output.")

# DINO_DIM = sorted(sample_dims)[0] if sample_dims else None
# print("DINO_DIM inferred as:", DINO_DIM)


## 6. Labels, matched to embeddings by StudyInstanceUID (UNCHANGED)

In [31]:
labels_df_raw = pd.read_csv(CONFIG["LABELS_CSV"])

ID_COL = CONFIG["ID_COL"]
label_cols = [c for c in labels_df_raw.columns if c != ID_COL]

print("Detected label columns:", label_cols)
if len(label_cols) != CONFIG["NUM_CLASSES"]:
    print(f"WARNING: expected {CONFIG['NUM_CLASSES']} label columns, found {len(label_cols)}.")
    CONFIG["NUM_CLASSES"] = len(label_cols)

labels_df_raw[ID_COL] = labels_df_raw[ID_COL].astype(str)
labels_df = labels_df_raw.set_index(ID_COL)


def get_label_and_mask(study_id):
    row = labels_df.loc[study_id, label_cols].values.astype(np.float32)
    mask = (~np.isnan(row)).astype(np.float32)
    target = np.nan_to_num(row, nan=0.0)
    return target, mask


Detected label columns: ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture', 'ACL__conf', 'MCL__conf', 'Medial Meniscus__conf', 'Lateral Meniscus__conf', 'Medial OA__conf', 'Lateral OA__conf', 'PF OA__conf', 'Effusion__conf', 'Synovitis__conf', "Baker's__conf", 'Contusion__conf', 'Fracture__conf']


## 6a. Official gold labels from the original competition `train.csv` (NEW)

**[GOLD-VAL MOD]** The soft/report-derived labels loaded above (`labels_df`) remain
the **training** targets -- nothing about that loading or `get_label_and_mask()`
changes. This section separately loads the *original* competition `train.csv` and
pulls out the studies that have **complete official binary labels** for all 12
abnormalities -- the "gold" studies, used **only for validation**, never for
training.


In [40]:
# ============================================================
# [GOLD-VAL MOD] LOAD OFFICIAL COMPETITION GOLD LABELS
# ============================================================

# IMPORTANT:
# `label_cols` belongs to the soft-label dataset and currently
# contains both the 12 targets AND their __conf columns.
#
# DO NOT modify label_cols.
#
# The official competition train.csv contains only the 12
# binary target columns, so define a separate list for gold
# validation.

GOLD_LABEL_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

assert len(GOLD_LABEL_COLS) == CONFIG["NUM_CLASSES"], (
    f"Expected {CONFIG['NUM_CLASSES']} gold labels, "
    f"got {len(GOLD_LABEL_COLS)}"
)


# ------------------------------------------------------------
# Load ORIGINAL competition train.csv
# ------------------------------------------------------------

competition_train_df = pd.read_csv(
    CONFIG["COMPETITION_TRAIN_CSV"]
)

competition_train_df[ID_COL] = (
    competition_train_df[ID_COL].astype(str)
)

print("Competition train.csv rows:", len(competition_train_df))
print("Competition train.csv columns:")
print(list(competition_train_df.columns))


# ------------------------------------------------------------
# Verify official target columns
# ------------------------------------------------------------

missing_gold_cols = [
    c
    for c in GOLD_LABEL_COLS
    if c not in competition_train_df.columns
]

if missing_gold_cols:
    raise ValueError(
        "Competition train.csv is missing official target "
        f"columns: {missing_gold_cols}"
    )


# ------------------------------------------------------------
# Identify the 58 GOLD studies
# ------------------------------------------------------------
#
# Gold studies are the rows where ALL 12 official competition
# labels are present.
#
# Do NOT use the soft-label dataframe here.
# Do NOT use the __conf columns here.
# ------------------------------------------------------------

gold_mask = (
    competition_train_df[GOLD_LABEL_COLS]
    .notna()
    .all(axis=1)
)

gold_df = competition_train_df.loc[
    gold_mask,
    [ID_COL] + GOLD_LABEL_COLS
].copy()

gold_df = gold_df.set_index(ID_COL)


# ------------------------------------------------------------
# Verify exactly 58 gold studies
# ------------------------------------------------------------

print(
    f"\nRows with complete official labels (gold): "
    f"{len(gold_df)}"
)

assert len(gold_df) == 58, (
    f"Expected exactly 58 gold studies, "
    f"found {len(gold_df)}. "
    f"Check CONFIG['COMPETITION_TRAIN_CSV']."
)


# ------------------------------------------------------------
# Verify gold labels are strictly binary
# ------------------------------------------------------------

gold_values = gold_df[GOLD_LABEL_COLS].values

gold_is_binary = (
    not np.isnan(gold_values).any()
    and np.all(np.isin(gold_values, [0, 1]))
)

print(
    "Gold label values are strictly binary (0/1):",
    gold_is_binary
)

assert gold_is_binary, (
    "Official gold labels contain values other than "
    "0/1 or contain NaN."
)


# ------------------------------------------------------------
# Gold IDs
# ------------------------------------------------------------

gold_ids = set(
    gold_df.index.astype(str)
)


# ------------------------------------------------------------
# Check embedding coverage
# ------------------------------------------------------------

embedded_ids = set(
    embeddings_index.keys()
)

missing_gold_embeddings = sorted(
    gold_ids - embedded_ids
)

if missing_gold_embeddings:

    print(
        f"WARNING: {len(missing_gold_embeddings)} "
        "gold studies have NO embeddings:"
    )

    for sid in missing_gold_embeddings:
        print(" ", sid)

else:

    print(
        "All 58 gold studies have embeddings."
    )


# ------------------------------------------------------------
# Gold label distribution
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GOLD LABEL DISTRIBUTION")
print("=" * 60)

for name in GOLD_LABEL_COLS:

    positives = int(
        gold_df[name].sum()
    )

    negatives = len(gold_df) - positives

    print(
        f"{name:25s} "
        f"positive: {positives:2d} | "
        f"negative: {negatives:2d} | "
        f"total: {len(gold_df)}"
    )


# ------------------------------------------------------------
# Gold-label lookup
# ------------------------------------------------------------

def get_gold_label(study_id):

    study_id = str(study_id)

    if study_id not in gold_df.index:
        raise KeyError(
            f"{study_id} is not present in gold_df"
        )

    return (
        gold_df.loc[
            study_id,
            GOLD_LABEL_COLS
        ]
        .values
        .astype(np.float32)
    )


print("\nGold studies:", len(gold_ids))
print("Gold classes:", len(GOLD_LABEL_COLS))

AssertionError: Expected 24 gold labels, got 12

## 7. Train/validation split: soft-labelled training studies vs. the 58 gold studies

**[GOLD-VAL MOD] Replaced** the previous random 80/20 split entirely. The
validation set is now **exactly the 58 official gold studies** identified in
Section 6a -- never randomly sampled, never thresholded, never re-split. The
training set is every embedded, soft-labelled study **except** those 58 gold
studies, which are held out for validation only and must never be trained on.


In [33]:
# [GOLD-VAL MOD] Replaced the previous random 80/20 split. Validation is now
# EXACTLY the 58 gold studies from Section 6a; training is every embedded,
# soft-labelled study EXCEPT those 58 (even if a gold study also happens to
# appear in the soft-label CSV, it is excluded from train_ids).
ID_COL = CONFIG["ID_COL"]

soft_label_ids = set(labels_df.index.astype(str))
embedded_ids = set(embeddings_index.keys())

val_ids = sorted(embedded_ids & gold_ids)
train_ids = sorted((embedded_ids & soft_label_ids) - gold_ids)

print("Studies with embeddings:", len(embedded_ids))
print("Studies with soft labels:", len(soft_label_ids))
print("Soft-labelled training studies:", len(train_ids))
print("Gold validation studies:", len(val_ids))

assert len(val_ids) == 58, f"Expected exactly 58 gold validation studies, got {len(val_ids)}"
assert len(set(train_ids) & set(val_ids)) == 0, "Gold studies leaked into the training set!"
print("Train/validation overlap:", len(set(train_ids) & set(val_ids)))

# Save the exact split for reproducibility
pd.DataFrame({ID_COL: train_ids}).to_csv(
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_train_ids.csv"),
    index=False
)
pd.DataFrame({ID_COL: val_ids}).to_csv(
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MIL_val_ids.csv"),
    index=False
)

print("No train/validation leakage.")


NameError: name 'gold_ids' is not defined

In [34]:
# [GOLD-VAL MOD] New cell -- dedicated label/split sanity check, run before training.
print("=" * 50)
print("LABEL / SPLIT SANITY CHECK")
print("=" * 50)
print()
print("Soft-label dataframe studies:", len(labels_df))
print("Gold studies from train.csv:", len(gold_df))
print()
print("Training studies:", len(train_ids))
print("Validation studies:", len(val_ids))
print()
print("Train/validation overlap:", len(set(train_ids) & set(val_ids)))
print()
print("Gold labels binary:", gold_is_binary)
print("Gold studies missing embeddings:", len(missing_gold_embeddings))
print()
print("-" * 50)
print("GOLD LABEL DISTRIBUTION")
print("-" * 50)
for name in label_cols:
    pos = int(gold_df[name].sum())
    print(f"{name:25s} positives: {pos} / {len(gold_df)}")


LABEL / SPLIT SANITY CHECK

Soft-label dataframe studies: 4407


NameError: name 'gold_df' is not defined

In [35]:
# Calculate per-task class weights using training data only.
# [GOLD-VAL MOD] Y_train now contains CONTINUOUS soft labels (train_ids excludes
# the 58 gold studies and uses only soft/report-derived labels), so the previous
# "Y_train == 1" / "Y_train == 0" hard counts would silently treat almost every
# sample as neither positive nor negative (soft labels are rarely exactly 1.0 or
# exactly 0.0) and produce near-meaningless weights. Weighting is now based on
# the soft-label MASS instead of hard counts:
#   positive_mass = sum of soft-label values     (how much "positive-ness" exists)
#   negative_mass = sum of (1 - soft-label value) (how much "negative-ness" exists)
# which reduces to the original exact behavior whenever labels really are 0/1.
# The positive-mass floor changed from max(positive_counts, 1) to
# max(positive_mass, 1e-6): a hard floor of 1 would badly distort the ratio for a
# continuous mass that can legitimately be e.g. 0.3; a tiny epsilon only guards
# against division by zero without changing the ratio otherwise. The existing
# [1.0, 5.0] cap is unchanged.
train_targets, train_masks = zip(
    *(get_label_and_mask(sid) for sid in train_ids)
)

Y_train = np.stack(train_targets).astype(np.float32)
M_train = np.stack(train_masks).astype(np.float32)

positive_mass = (Y_train * M_train).sum(axis=0)
negative_mass = ((1.0 - Y_train) * M_train).sum(axis=0)

# Standard positive weighting: negative mass / positive mass
raw_pos_weights = negative_mass / np.maximum(positive_mass, 1e-6)

# Conservative initial experiment: only boost positives,
# and cap very large weights to reduce instability. (cap policy UNCHANGED)
pos_weights = np.clip(raw_pos_weights, 1.0, 5.0)

POS_WEIGHTS = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device
)

for i, name in enumerate(label_cols):
    print(
        f"{name:25s} | "
        f"positive_mass={positive_mass[i]:8.2f} | "
        f"negative_mass={negative_mass[i]:8.2f} | "
        f"pos_weight={pos_weights[i]:.3f}"
    )


ACL                       | positive_mass=  836.16 | negative_mass= 2688.87 | pos_weight=3.216
MCL                       | positive_mass=  629.17 | negative_mass= 2895.90 | pos_weight=4.603
Medial Meniscus           | positive_mass= 1616.50 | negative_mass= 1908.50 | pos_weight=1.181
Lateral Meniscus          | positive_mass=  786.97 | negative_mass= 2738.07 | pos_weight=3.479
Medial OA                 | positive_mass= 1432.42 | negative_mass= 2092.58 | pos_weight=1.461
Lateral OA                | positive_mass= 1149.84 | negative_mass= 2375.18 | pos_weight=2.066
PF OA                     | positive_mass= 1708.21 | negative_mass= 1816.79 | pos_weight=1.064
Effusion                  | positive_mass= 2069.36 | negative_mass= 1455.65 | pos_weight=1.000
Synovitis                 | positive_mass= 1195.20 | negative_mass= 2329.83 | pos_weight=1.949
Baker's                   | positive_mass= 1252.87 | negative_mass= 2272.13 | pos_weight=1.814
Contusion                 | positive_mass=  927.22

## 8. Batch construction (padding + masking UNCHANGED; `.pin_memory()` added)

The padding/masking logic itself is identical to the original notebook. The only
change (`# [MULTI-HEAD MOD]` below) is pinning the collated CPU tensors so the later
`.to(device, non_blocking=True)` transfers in the training/validation loops are
actually asynchronous instead of silently falling back to a synchronous copy.


In [36]:
def collate_items(items, embed_dim):
    B = len(items)
    max_series = max(max(len(it[1]), 1) for it in items)
    max_windows = max(max((arr.shape[0] for arr in it[1]), default=1) for it in items)
    max_windows = max(max_windows, 1)

    window_features = np.zeros((B, max_series, max_windows, embed_dim), dtype=np.float32)
    window_mask = np.zeros((B, max_series, max_windows), dtype=np.float32)
    series_mask = np.zeros((B, max_series), dtype=np.float32)
    targets = np.zeros((B, len(items[0][2])), dtype=np.float32)
    label_masks = np.zeros((B, len(items[0][3])), dtype=np.float32)
    study_ids = []

    for i, (study_id, series_arrays, target, label_mask) in enumerate(items):
        study_ids.append(study_id)
        targets[i] = target
        label_masks[i] = label_mask
        for s, arr in enumerate(series_arrays):
            n_win = arr.shape[0]
            window_features[i, s, :n_win, :] = arr
            window_mask[i, s, :n_win] = 1.0
            series_mask[i, s] = 1.0

    def _to_tensor_pinned(np_arr):
        t = torch.from_numpy(np_arr)
        # [MULTI-HEAD MOD] pin_memory() for faster, truly-async H2D copies; only
        # useful (and only valid) when CUDA is actually available.
        if torch.cuda.is_available():
            t = t.pin_memory()
        return t

    return {
        "study_ids": study_ids,
        "window_features": _to_tensor_pinned(window_features),
        "window_mask": _to_tensor_pinned(window_mask),
        "series_mask": _to_tensor_pinned(series_mask),
        "targets": _to_tensor_pinned(targets),
        "label_masks": _to_tensor_pinned(label_masks),
    }


def build_batch(study_id_batch, embeddings_index, embed_dim):
    items = []
    n_no_embedding = 0
    for sid in study_id_batch:
        series_dict = embeddings_index.get(sid, {})
        series_arrays = [arr for arr in series_dict.values() if arr is not None and arr.shape[0] > 0]
        if len(series_arrays) == 0:
            n_no_embedding += 1
        target, mask = get_label_and_mask(sid)
        items.append((sid, series_arrays, target, mask))
    return collate_items(items, embed_dim), n_no_embedding


## 9. Model: three-stage independent specialists (projection -> masked mean pool -> task-specific series attention -> single linear classifier)

**[ARCHITECTURE MOD]** Replaced the previous per-task hierarchical
Window-MIL-attention -> Series-MIL-attention -> multi-layer MLP with a simpler
three-stage design, per task. The frozen DINOv3 embeddings remain the single
shared input; everything downstream of them is still fully independent per
abnormality (its own projection, its own series attention, its own classifier) --
no trainable parameters are shared between tasks, consistent with Section 12's
"no shared trainable parameters" check below.

**New per-task architecture:**

`Frozen DINOv3 embeddings -> task-specific projection -> masked mean pooling over windows -> task-specific gated series attention -> single Linear(512,1) classifier -> one logit`

**Dimension flow for one study** (`D` = detected `DINO_DIM`, `PROJECTED_DIM` = 512 by default, `S`/`W` = that study's actual series/window counts):

| Stage | Tensor shape |
|---|---|
| DINOv3 embedding input | `[S, W, D]` |
| Projection (Linear + GELU + LayerNorm) | `[S, W, 512]` |
| Masked mean pooling over windows | `[S, 512]` |
| Attention scores (per task) | `[S]` |
| Attention weights (per task, masked softmax) | `[S]` |
| Attention-pooled study representation (per task) | `[512]` |
| Classifier output (per task) | `[1]` |
| Combined output (12 tasks concatenated) | `[12]` |

What changed vs. what didn't:
- **Window-level pooling**: was a learned gated-attention MIL module per task; now
  parameter-free masked mean pooling (`masked_mean_pool` below) -- padded windows
  contribute exactly zero to the average, and a fully-padded series still produces
  a finite (not NaN) vector, since the series-level mask zeroes it out at the next
  stage anyway.
- **Series-level pooling**: unchanged -- still the same gated-attention mechanism
  (`score = w(tanh(V(x)) * sigmoid(U(x)))`, masked softmax over valid series),
  reusing the existing `MaskedAttentionMIL` class as-is, just no longer also used
  at the window level.
- **Classifier**: was a multi-layer MLP (`head_hidden_dims`, dropout); now a single
  `Linear(512, 1)` with no hidden layers, no activation, no dropout.
- **Per-task independence**: unchanged in spirit (each task still has its own
  projection, its own attention module, its own classifier, its own optimizer and
  AMP scaler -- see Sections 11/16). An independent *per-task* projection was kept
  (rather than one shared projection) specifically so the existing fully-independent
  per-task optimizers remain correct: a shared projection would otherwise be
  stepped by 12 different optimizers independently, which is exactly the kind of
  inconsistent shared-parameter update this notebook's training loop is designed
  to avoid.


In [37]:
# class MaskedAttentionMIL(nn.Module):
#     # UNCHANGED from the original notebook.
#     def __init__(self, in_dim=512, hidden_dim=128):
#         super().__init__()
#         self.attn_V = nn.Linear(in_dim, hidden_dim)
#         self.attn_U = nn.Linear(in_dim, hidden_dim)
#         self.attn_w = nn.Linear(hidden_dim, 1)

#     def forward(self, x, mask):
#         A_V = torch.tanh(self.attn_V(x))
#         A_U = torch.sigmoid(self.attn_U(x))
#         scores = self.attn_w(A_V * A_U).squeeze(-1)

#         neg_inf = torch.finfo(scores.dtype).min
#         scores = scores.masked_fill(mask == 0, neg_inf)

#         all_masked = mask.sum(dim=-1, keepdim=True) == 0
#         scores = torch.where(all_masked.expand_as(scores), torch.zeros_like(scores), scores)

#         attn = torch.softmax(scores, dim=-1)
#         attn = attn * mask
#         denom = attn.sum(dim=-1, keepdim=True).clamp(min=1e-8)
#         attn = attn / denom

#         pooled = torch.sum(attn.unsqueeze(-1) * x, dim=-2)
#         return pooled, attn


# class EmbeddingProjection(nn.Module):
#     # UNCHANGED from the original notebook.
#     def __init__(self, in_dim, out_dim=512):
#         super().__init__()
#         self.proj = nn.Linear(in_dim, out_dim)
#         self.act = nn.GELU()
#         self.norm = nn.LayerNorm(out_dim)

#     def forward(self, x):
#         return self.norm(self.act(self.proj(x)))


# # [MULTI-HEAD MOD] New: one independent MLP per abnormality.
# class IndependentTaskHead(nn.Module):
#     """512 -> 128 -> 64 -> 1 with ReLU + dropout. One instance per abnormality;
#     each head has its own weights (no sharing), so each abnormality gets a
#     dedicated decision boundary instead of sharing the final linear layer."""

#     def __init__(self, in_dim, hidden_dims=(128, 64), dropout=0.2):
#         super().__init__()
#         h1, h2 = hidden_dims
#         self.net = nn.Sequential(
#             nn.Linear(in_dim, h1),
#             nn.ReLU(inplace=True),
#             nn.Dropout(dropout),
#             nn.Linear(h1, h2),
#             nn.ReLU(inplace=True),
#             nn.Dropout(dropout),
#             nn.Linear(h2, 1),
#         )

#     def forward(self, x):
#         return self.net(x)  # [B, 1], raw logit -- no sigmoid here


# # [MULTI-HEAD MOD] New: replaces the old shared nn.Sequential(512,128,12) head.
# class MultiTaskHeads(nn.Module):
#     """12 independent IndependentTaskHead modules sharing the same input
#     (the study-level MIL representation), trained jointly in a single forward/
#     backward pass. Outputs are concatenated along dim=1 into (batch_size, 12),
#     matching the shape the rest of the pipeline (loss, AUC, checkpoints) expects
#     -- so nothing downstream needed to change shape-wise."""

#     def __init__(self, in_dim, num_classes, hidden_dims=(128, 64), dropout=0.2):
#         super().__init__()
#         self.heads = nn.ModuleList([
#             IndependentTaskHead(in_dim, hidden_dims, dropout) for _ in range(num_classes)
#         ])

#     def forward(self, x):
#         # Each head is a small MLP; looping over 12 of them is negligible next to
#         # the MIL attention / projection cost, so no custom batched-head kernel
#         # is needed here -- this is still one Python-level forward/backward call.
#         outputs = [head(x) for head in self.heads]  # list of [B, 1]
#         return torch.cat(outputs, dim=1)             # [B, num_classes]


# class OrthoDinoMILModel(nn.Module):
#     def __init__(self, dino_dim, projected_dim, attn_hidden, num_classes,
#                  head_hidden_dims=(128, 64), head_dropout=0.2):
#         super().__init__()
#         self.projection = EmbeddingProjection(dino_dim, projected_dim)
#         self.window_mil = MaskedAttentionMIL(projected_dim, attn_hidden)
#         self.series_mil = MaskedAttentionMIL(projected_dim, attn_hidden)
#         # [MULTI-HEAD MOD] was: nn.Sequential(Linear(512,128), ReLU, Linear(128,12))
#         self.head = MultiTaskHeads(projected_dim, num_classes, head_hidden_dims, head_dropout)

#     def forward(self, window_features, window_mask, series_mask):
#         projected = self.projection(window_features)                    # [B,S,W,512]
#         series_vecs, _ = self.window_mil(projected, window_mask)        # [B,S,512]
#         study_vec, _ = self.series_mil(series_vecs, series_mask)        # [B,512] -- shared representation
#         logits = self.head(study_vec)                                   # [B,12]  -- 12 independent heads
#         return logits


# assert DINO_DIM is not None, "Could not infer DINO_DIM from the embeddings -- check Section 4/5 output."

# model = OrthoDinoMILModel(
#     DINO_DIM, CONFIG["PROJECTED_DIM"], CONFIG["ATTENTION_HIDDEN_DIM"], CONFIG["NUM_CLASSES"],
#     CONFIG["MULTIHEAD_HIDDEN_DIMS"], CONFIG["MULTIHEAD_DROPOUT"],
# ).to(device)

# print(model)
# print(f"\nShared representation dimension feeding the 12 heads: {CONFIG['PROJECTED_DIM']}")
# print(f"Per-head architecture: {CONFIG['PROJECTED_DIM']} -> {CONFIG['MULTIHEAD_HIDDEN_DIMS'][0]} "
#       f"-> {CONFIG['MULTIHEAD_HIDDEN_DIMS'][1]} -> 1  (x{CONFIG['NUM_CLASSES']} independent heads)")


In [38]:

# class MaskedAttentionMIL(nn.Module):
#     # UNCHANGED gated-attention mechanism. Previously used at both the window
#     # level and the series level per task; now used only for series-level
#     # pooling (window-level pooling is masked_mean_pool, below).
#     def __init__(self, in_dim=512, hidden_dim=128):
#         super().__init__()
#         self.attn_V = nn.Linear(in_dim, hidden_dim)
#         self.attn_U = nn.Linear(in_dim, hidden_dim)
#         self.attn_w = nn.Linear(hidden_dim, 1)

#     def forward(self, x, mask):
#         scores = self.attn_w(
#             torch.tanh(self.attn_V(x)) *
#             torch.sigmoid(self.attn_U(x))
#         ).squeeze(-1)

#         scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
#         all_masked = mask.sum(dim=-1, keepdim=True) == 0
#         scores = torch.where(all_masked, torch.zeros_like(scores), scores)

#         attn = torch.softmax(scores, dim=-1) * mask
#         attn = attn / attn.sum(dim=-1, keepdim=True).clamp(min=1e-8)
#         pooled = torch.sum(attn.unsqueeze(-1) * x, dim=-2)
#         return pooled, attn


# class EmbeddingProjection(nn.Module):
#     # UNCHANGED.
#     def __init__(self, in_dim, out_dim=512):
#         super().__init__()
#         self.proj = nn.Linear(in_dim, out_dim)
#         self.act = nn.GELU()
#         self.norm = nn.LayerNorm(out_dim)

#     def forward(self, x):
#         return self.norm(self.act(self.proj(x)))


# # [ARCHITECTURE MOD] New: parameter-free replacement for the old window-level
# # MaskedAttentionMIL. Mean-pools over the window dimension, ignoring padded
# # windows entirely. A fully-padded series (all windows masked out) returns a
# # finite vector (division is clamped), never NaN -- safe because such a series
# # is also zeroed out by series_mask at the next (series-attention) stage.
# def masked_mean_pool(x, mask):
#     """x: [..., W, D], mask: [..., W] (1 = real window, 0 = pad) -> [..., D]"""
#     mask = mask.unsqueeze(-1)                      # [..., W, 1]
#     summed = (x * mask).sum(dim=-2)                # [..., D]
#     counts = mask.sum(dim=-2).clamp(min=1e-8)       # [..., 1]
#     return summed / counts


# class IndependentTaskModel(nn.Module):
#     """One fully independent specialist for one abnormality:
#     projection -> masked mean pooling over windows -> task-specific gated
#     attention over series -> single Linear(512, 1) classifier.

#     [ARCHITECTURE MOD] Previously: projection -> window-level gated-attention
#     MIL -> series-level gated-attention MIL -> multi-layer MLP classifier
#     (head_hidden_dims + dropout). The window-level attention is replaced with
#     masked_mean_pool, and the classifier is now a single linear layer -- no
#     hidden layers, no activation, no dropout -- per the simplified three-stage
#     architecture. Series-level attention is unchanged."""

#     def __init__(self, dino_dim, projected_dim=512, attn_hidden=128):
#         super().__init__()
#         self.projection = EmbeddingProjection(dino_dim, projected_dim)
#         # [ARCHITECTURE MOD] was: self.window_mil = MaskedAttentionMIL(...)
#         # Window-level pooling is now parameter-free (masked_mean_pool above),
#         # not a learned attention module -- intentionally no self.window_mil.
#         self.series_attention = MaskedAttentionMIL(projected_dim, attn_hidden)
#         # [ARCHITECTURE MOD] was: a multi-layer nn.Sequential MLP built from
#         # head_hidden_dims with ReLU + Dropout between layers. Now a single
#         # linear layer with no hidden layers.
#         self.classifier = nn.Linear(projected_dim, 1)

#     def forward(self, window_features, window_mask, series_mask):
#         x = self.projection(window_features)                      # [B, S, W, 512]
#         series_vecs = masked_mean_pool(x, window_mask)             # [B, S, 512]
#         study_vec, _ = self.series_attention(series_vecs, series_mask)  # [B, 512]
#         return self.classifier(study_vec).squeeze(-1)              # [B]


# class OrthoDinoMILModel(nn.Module):
#     # [ARCHITECTURE MOD] Dropped head_hidden_dims/head_dropout parameters --
#     # the new classifier is a fixed single Linear(512,1) with nothing to
#     # configure. All call sites that construct this class were updated to
#     # match (Sections 9, 17, 18).
#     def __init__(self, dino_dim, projected_dim, attn_hidden, num_classes):
#         super().__init__()
#         self.tasks = nn.ModuleList([
#             IndependentTaskModel(dino_dim, projected_dim, attn_hidden)
#             for _ in range(num_classes)
#         ])

#     def forward_task(self, task_idx, window_features, window_mask, series_mask):
#         return self.tasks[task_idx](window_features, window_mask, series_mask)

#     def forward(self, window_features, window_mask, series_mask):
#         # Outputs are concatenated for evaluation/inference only.
#         logits = [
#             self.forward_task(i, window_features, window_mask, series_mask)
#             for i in range(len(self.tasks))
#         ]
#         return torch.stack(logits, dim=1)


# model = OrthoDinoMILModel(
#     DINO_DIM,
#     CONFIG["PROJECTED_DIM"],
#     CONFIG["ATTENTION_HIDDEN_DIM"],
#     CONFIG["NUM_CLASSES"],
# ).to(device)

# print(model)
# print("Independent specialist branches:", len(model.tasks))


In [39]:

class EmbeddingProjection(nn.Module):
    """Project frozen DINOv3 embeddings into a trainable feature space."""
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.norm(self.act(self.proj(x)))


def masked_mean_pool(x, mask):
    """
    x:    [B, ..., N, D]
    mask: [B, ..., N]
    Returns: [B, ..., D]
    """
    mask = mask.unsqueeze(-1).to(dtype=x.dtype)
    summed = (x * mask).sum(dim=-2)
    counts = mask.sum(dim=-2).clamp(min=1e-8)
    return summed / counts


class IndependentTaskModel(nn.Module):
    """One independent specialist for a single abnormality."""

    def __init__(self, dino_dim, projected_dim=512):
        super().__init__()

        # Independent feature projection for this task
        self.projection = EmbeddingProjection(
            dino_dim, projected_dim
        )

        # Task-specific neural network
        self.classifier = nn.Sequential(
            nn.Linear(projected_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),

            nn.Linear(128, 1)
        )

    def forward(self, window_features, window_mask, series_mask):
        # [B, S, W, D] -> [B, S, W, 512]
        x = self.projection(window_features)

        # Masked mean over windows -> [B, S, 512]
        series_features = masked_mean_pool(x, window_mask)

        # Exclude series with no valid windows
        valid_series = (
            (series_mask > 0) & (window_mask.sum(dim=-1) > 0)
        ).to(dtype=x.dtype)

        # Masked mean over series -> [B, 512]
        study_features = masked_mean_pool(
            series_features, valid_series
        )

        # [B, 512] -> [B, 1] -> [B]
        logits = self.classifier(study_features).squeeze(-1)
        return logits


class OrthoDinoMILModel(nn.Module):
    """12 independent specialists, one for each abnormality."""

    def __init__(
        self, dino_dim, projected_dim=512,
        attn_hidden=None, num_classes=12
    ):
        super().__init__()

        # attn_hidden retained for compatibility with existing call sites.
        self.tasks = nn.ModuleList([
            IndependentTaskModel(dino_dim, projected_dim)
            for _ in range(num_classes)
        ])

    def forward_task(
        self, task_idx, window_features, window_mask, series_mask
    ):
        return self.tasks[task_idx](
            window_features, window_mask, series_mask
        )

    def forward(self, window_features, window_mask, series_mask):
        # Each specialist outputs [B]; combined output is [B, 12]
        logits = [
            task(window_features, window_mask, series_mask)
            for task in self.tasks
        ]
        return torch.stack(logits, dim=1)


model = OrthoDinoMILModel(
    DINO_DIM,
    CONFIG["PROJECTED_DIM"],
    CONFIG["ATTENTION_HIDDEN_DIM"],
    CONFIG["NUM_CLASSES"]
).to(device)

print(model)
print("Independent specialists:", len(model.tasks))


OrthoDinoMILModel(
  (tasks): ModuleList(
    (0-23): 24 x IndependentTaskModel(
      (projection): EmbeddingProjection(
        (proj): Linear(in_features=1024, out_features=512, bias=True)
        (act): GELU(approximate='none')
        (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
      (classifier): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.25, inplace=False)
        (4): Linear(in_features=256, out_features=128, bias=True)
        (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (6): GELU(approximate='none')
        (7): Dropout(p=0.15, inplace=False)
        (8): Linear(in_features=128, out_features=1, bias=True)
      )
    )
  )
)
Independent specialists: 24


In [ ]:

def classifier_l1_penalty(task_model):
    """
    L1 penalty on classifier weights only.
    Biases are excluded.
    """
    l1 = torch.zeros(
        (),
        device=next(task_model.parameters()).device
    )

    for module in task_model.classifier.modules():
        if isinstance(module, nn.Linear):
            l1 = l1 + module.weight.abs().sum()

    return l1


### 9a. Synthetic verification of the new architecture (no Kaggle dataset or GPU required)

**[ARCHITECTURE MOD] New cell.** Before touching real data, this runs the new
three-stage model on small random tensors with deliberately-placed padding to
check, with actual assertions rather than claims: output shape, that masked mean
pooling truly ignores padded windows and never produces NaN on a fully-padded
series, that series-attention weights sum to 1 over valid series and are exactly
0 on padded series, that `classifier_l1_penalty` still works against the new
single-`Linear` classifier, and -- the independent-training requirement -- that
calling `backward()` on one task's loss produces gradients _only_ for that
task's own parameters and leaves every other task's weights untouched after an
optimizer step. These checks were run in a CPU-only sandbox against this exact
code (not just the real Kaggle run) and all passed; see the printed output below
once you run this cell.


In [ ]:
# [ARCHITECTURE MOD] New verification cell -- synthetic data only, no GPU/dataset
# required. Mirrors the checks already run against this exact architecture code
# in a CPU-only sandbox before this notebook was returned (all passed there too).
with torch.no_grad():
    _B, _S, _W = 3, 4, 5
    torch.manual_seed(0)
    _wf = torch.randn(_B, _S, _W, DINO_DIM, device=device)
    _wm = torch.ones(_B, _S, _W, device=device)
    _sm = torch.ones(_B, _S, device=device)

    # Study 0: series 2 and 3 are fully padded (no real series at all).
    _sm[0, 2:] = 0
    _wm[0, 2:, :] = 0
    # Study 0, series 1: only the first 2 of 5 windows are real.
    _wm[0, 1, 2:] = 0
    # Study 1, series 0: window 0 is padded but the rest of that series is real.
    _wm[1, 0, 0] = 0

    _task0 = model.tasks[0]
    _logits = _task0(_wf, _wm, _sm)
    assert _logits.shape == (_B,), f"Expected per-task logits shape (B,), got {tuple(_logits.shape)}"

    _proj = _task0.projection(_wf)
    _pooled_windows = masked_mean_pool(_proj, _wm)
    assert torch.isfinite(_pooled_windows).all(), \
        "masked_mean_pool produced non-finite values on a fully-padded series"
    _manual_mean = _proj[1, 0, 1:, :].mean(dim=0)
    assert torch.allclose(_pooled_windows[1, 0], _manual_mean, atol=1e-4), \
        "masked_mean_pool is including a padded window in the average"

    _study_vec, _attn = _task0.series_attention(_pooled_windows, _sm)
    _attn_sums = _attn.sum(dim=-1)
    assert torch.allclose(_attn_sums, torch.ones(_B, device=device), atol=1e-4), \
        f"Attention weights do not sum to 1: {_attn_sums}"
    assert torch.all(_attn[0, 2:] == 0), "Padded series received nonzero attention weight"
    assert torch.isfinite(_study_vec).all(), "Series attention produced non-finite output"

print("Synthetic forward-pass checks passed: output shape, masked mean pooling "
      "(padded-window exclusion + fully-padded-series NaN-safety), and series "
      "attention (sums to 1, zero weight on padding) all verified on random data.")

# Independent-gradient check: backward() on task 0's loss only must not affect
# task 1's parameters, and an optimizer step on task 0 must not move task 1 at all.
model.zero_grad(set_to_none=True)
_wf2 = torch.randn(2, 2, 3, DINO_DIM, device=device)
_wm2 = torch.ones(2, 2, 3, device=device)
_sm2 = torch.ones(2, 2, device=device)
_targets2 = torch.randint(0, 2, (2,), device=device).float()

_opt0_check = torch.optim.AdamW(model.tasks[0].parameters(), lr=1e-3)
_before_t1 = [p.detach().clone() for p in model.tasks[1].parameters()]

_logits_t0 = model.forward_task(0, _wf2, _wm2, _sm2)
_loss_t0 = F.binary_cross_entropy_with_logits(_logits_t0, _targets2)
_opt0_check.zero_grad(set_to_none=True)
_loss_t0.backward()

_task0_has_grad = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in model.tasks[0].parameters())
_task1_has_grad = any(p.grad is not None for p in model.tasks[1].parameters())
assert _task0_has_grad, "Task 0 received no gradient from its own loss -- backward pass is broken"
assert not _task1_has_grad, "Task 1 received a gradient from task 0's backward() -- branches are NOT independent!"

_opt0_check.step()
_after_t1 = [p.detach().clone() for p in model.tasks[1].parameters()]
assert all(torch.equal(a, b) for a, b in zip(_before_t1, _after_t1)), \
    "Task 1's weights changed after task 0's optimizer.step() -- parameters are shared!"

model.zero_grad(set_to_none=True)
print("Independent-gradient check passed: task 0's backward() + optimizer step "
      "produced gradients/updates ONLY on task 0's own parameters; task 1 was "
      "completely untouched.")


## 10. Optional Stage 1 initialization (Option A) vs. fresh init (Option B, default)

**[MULTI-HEAD MOD]** Window MIL / Series MIL loading from Stage 1 is unchanged and
still works (same shapes as before). Loading the Stage 1 **head** is no longer
possible: Stage 1's head is a single shared `512->128->12` block, while the new
architecture has 12 independent `512->128->64->1` heads -- the parameter shapes do
not match, so that load is skipped outright (with an explanation) rather than
attempted and failing, and the 12 new heads always start from fresh initialization
regardless of this flag.


In [ ]:
def find_stage1_file(filename, search_dirs):
    for d in search_dirs:
        direct = os.path.join(d, filename)
        if os.path.exists(direct):
            return direct
        matches = glob.glob(os.path.join(d, "**", filename), recursive=True)
        if matches:
            return matches[0]
    return None


if CONFIG["INIT_FROM_STAGE1"]:
    # [ARCHITECTURE MOD] Stage 1 produced ONE shared window-attention/series-attention
    # pair for a single shared-attention model. The current architecture has 12 fully
    # independent task branches, each with its own projection and its own series
    # attention module, and no window-level attention module at all anymore (window
    # pooling is now masked_mean_pool, which has no learnable weights to load into).
    # There is no sound way to map Stage 1's weights onto this architecture, so this
    # raises clearly instead of silently touching a nonexistent `model.window_mil`
    # (which would previously have failed with a confusing AttributeError).
    raise NotImplementedError(
        "INIT_FROM_STAGE1=True is not supported for the current independent-specialist "
        "three-stage architecture: there are 12 separate task branches (no single "
        "shared window/series attention module), and window-level pooling is now "
        "parameter-free masked mean pooling with nothing to initialize from Stage 1. "
        "Set CONFIG['INIT_FROM_STAGE1'] = False to train from scratch, which is the "
        "only supported path for this architecture."
    )
else:
    print("Fresh initialization (the only supported path for this architecture): "
          "each of the 12 independent task branches -- projection, series attention, "
          "and classifier -- starts from PyTorch's default initialization.")


## 11. Loss, optimizer, AMP scaler (UNCHANGED: same masked BCE, same AdamW/LR/weight decay)

In [ ]:
# def masked_bce_loss(logits, targets, label_masks):
#     # UNCHANGED. Still BCEWithLogits on raw logits (sigmoid is applied only for
#     # evaluation/inference in Sections 15-18, never here).
#     loss_per_elem = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
#     loss_per_elem = loss_per_elem * label_masks
#     denom = label_masks.sum().clamp(min=1.0)
#     return loss_per_elem.sum() / denom


# # UNCHANGED optimizer/LR/weight-decay config; now just built over the new model's
# # parameters (projection + MIL + the 12 new heads, all trained jointly).
# optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
# scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# print("Optimizer: AdamW, lr=", CONFIG["LR"], "weight_decay=", CONFIG["WEIGHT_DECAY"])
# print("Trainable parameter groups now include 12 independent heads instead of 1 shared head.")


In [ ]:

# def masked_task_bce_loss(logits, targets, label_masks, task_idx):
#     mask = label_masks[:, task_idx]
#     valid = mask > 0

#     if not valid.any():
#         return None

#     task_logits = logits[valid]
#     task_targets = targets[valid, task_idx]

#     return F.binary_cross_entropy_with_logits(
#         task_logits, task_targets
#     )

def masked_task_bce_loss(
    logits, targets, label_masks, task_idx
):
    mask = label_masks[:, task_idx] > 0

    if not mask.any():
        return None

    task_logits = logits[mask].float()
    task_targets = targets[mask, task_idx].float()

    return F.binary_cross_entropy_with_logits(
        task_logits,
        task_targets,
        pos_weight=POS_WEIGHTS[task_idx]
    )


# One optimizer and AMP scaler per abnormality.
optimizers = [
    torch.optim.AdamW(
        task.parameters(),
        lr=CONFIG["LR"],
        weight_decay=CONFIG["WEIGHT_DECAY"]
    )
    for task in model.tasks
]

scalers = [
    torch.cuda.amp.GradScaler(enabled=USE_AMP)
    for _ in model.tasks
]

print("Created", len(optimizers), "independent optimizers.")


## 12. Sanity checks (run before training) -- UNCHANGED logic, now validates the new model

In [ ]:
print("=" * 70)
print("SANITY CHECKS")
print("=" * 70)

print(f"Embedding files found:                 {len(all_embedding_files)}")
print(f"Studies with embeddings:                {len(embedded_ids)}")
print(f"Matched labeled studies (train+val):    {len(train_ids) + len(val_ids)}")
print(f"Train / Val sizes:                      {len(train_ids)} / {len(val_ids)}")
print(f"DINOv3 embedding dimension:              {DINO_DIM}")
print(f"Shared study representation dimension:   {CONFIG['PROJECTED_DIM']}")
# [ARCHITECTURE MOD] was: printed CONFIG['MULTIHEAD_HIDDEN_DIMS'], which no
# longer exists as part of the model (single Linear(512,1) classifier now).
print(f"Per-task architecture:                   DINO[{DINO_DIM}] -> "
      f"projection[{CONFIG['PROJECTED_DIM']}] -> masked mean pool over windows -> "
      f"gated series attention -> Linear({CONFIG['PROJECTED_DIM']},1) "
      f"(x{CONFIG['NUM_CLASSES']} fully independent branches)")

series_counts, window_counts = [], []
shape_anomalies = []
for sid in (train_ids + val_ids):
    sdict = embeddings_index.get(sid, {})
    series_counts.append(len(sdict))
    for arr in sdict.values():
        window_counts.append(arr.shape[0])
        if arr.shape[-1] != DINO_DIM:
            shape_anomalies.append((sid, arr.shape))

if series_counts:
    print(f"Series per study:   min={min(series_counts)} max={max(series_counts)} mean={np.mean(series_counts):.2f}")
if window_counts:
    print(f"Windows per series: min={min(window_counts)} max={max(window_counts)} mean={np.mean(window_counts):.2f}")

if shape_anomalies:
    print(f"WARNING: {len(shape_anomalies)} series have an embedding dim != {DINO_DIM}. First few: {shape_anomalies[:5]}")
else:
    print("All matched studies have the expected embedding dimensionality.")

# n_params = sum(p.numel() for p in model.parameters())
# n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# n_head_params = sum(p.numel() for p in model.head.parameters())
# print(f"Model total parameters:                  {n_params:,}")
# print(f"Model trainable parameters:              {n_trainable:,}")
# print(f"  of which in the 12 independent heads:  {n_head_params:,}")

# print(f"CUDA available:                          {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU:                                      {torch.cuda.get_device_name(0)}")
#     print(f"Free CUDA memory (approx):                "
#           f"{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / (1024**3):.2f} GB")


n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

for i, name in enumerate(label_cols):
    n = sum(p.numel() for p in model.tasks[i].parameters())
    print(f"{name}: {n:,} trainable parameters")

# Verify there are no shared trainable parameters.
param_ids = [
    {id(p) for p in task.parameters() if p.requires_grad}
    for task in model.tasks
]
for i in range(len(param_ids)):
    for j in range(i + 1, len(param_ids)):
        assert param_ids[i].isdisjoint(param_ids[j]), (
            f"Trainable parameters are shared between tasks {i} and {j}"
        )

print("Verified: no trainable parameters are shared.")


In [ ]:
def check_finite(model, sample_logits=None):
    ok = True
    for name, p in model.named_parameters():
        if not torch.isfinite(p).all():
            print(f"WARNING: parameter {name} contains non-finite values!")
            ok = False
    if sample_logits is not None and not torch.isfinite(sample_logits).all():
        print("WARNING: sample forward output contains non-finite values!")
        ok = False
    return ok


sanity_batch, _ = build_batch(train_ids[:4], embeddings_index, DINO_DIM)
wf = sanity_batch["window_features"].to(device)
wm = sanity_batch["window_mask"].to(device)
sm = sanity_batch["series_mask"].to(device)

model.eval()
with torch.no_grad():
    sanity_logits = model(wf, wm, sm)

print("Sanity forward pass output shape:", tuple(sanity_logits.shape))
# [MULTI-HEAD MOD] Shape check still asserts the SAME (batch, 12) contract as before --
# confirms the 12 independent heads concatenate correctly.
assert sanity_logits.shape[-1] == CONFIG["NUM_CLASSES"], "Model did not output exactly 12 logits!"
print("Confirmed: model outputs exactly", CONFIG["NUM_CLASSES"], "logits per study (12 independent heads, concatenated).")

finite_ok = check_finite(model, sanity_logits)
print("All parameters and sample outputs finite:", finite_ok)
assert finite_ok, "Non-finite values detected before training even started -- investigate before proceeding."

model.train()
print("=" * 70)


## 13. Checkpoint utilities

Same checkpoint schema/contents as the original notebook (projection/Window MIL/
Series MIL/head weights, optimizer and AMP scaler state, epoch and in-epoch batch
index, best validation AUC and its full history, the fixed train/val study-ID lists,
and RNG states). **[MULTI-HEAD MOD]**: `torch.load(..., weights_only=False)` is now
explicit on every load of a checkpoint this notebook itself produces, for PyTorch
2.6+ compatibility (where `weights_only=True` is the new default and would reject
the plain Python objects -- lists, dicts, config -- stored alongside the tensors).

**[GOLD-VAL MOD]**: checkpoints now also store `validation_protocol: "gold_58_v1"` and `n_gold_val_studies`, and `best_val_auc`/`val_auc` here mean the 58-study gold macro ROC-AUC, not a random-split AUC. Resuming a checkpoint saved under a different validation protocol now raises instead of silently mixing the two.


In [ ]:
# def get_rng_states():
#     state = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
#     if torch.cuda.is_available():
#         state["torch_cuda"] = torch.cuda.get_rng_state_all()
#     return state


# def set_rng_states(state):
#     try:
#         random.setstate(state["python"])
#         np.random.set_state(state["numpy"])
#         torch.set_rng_state(state["torch"])
#         if torch.cuda.is_available() and "torch_cuda" in state:
#             torch.cuda.set_rng_state_all(state["torch_cuda"])
#     except Exception as e:
#         print("Could not fully restore RNG state (non-fatal):", e)


# def save_checkpoint(path, epoch, resume_batch_index, val_auc, best_val_auc, val_auc_history):
#     ckpt = {
#         "projection_state_dict": model.projection.state_dict(),
#         "window_mil_state_dict": model.window_mil.state_dict(),
#         "series_mil_state_dict": model.series_mil.state_dict(),
#         "head_state_dict": model.head.state_dict(),  # now MultiTaskHeads' state_dict
#         "optimizer_state_dict": optimizer.state_dict(),
#         "scaler_state_dict": scaler.state_dict(),
#         "epoch": epoch,
#         "resume_batch_index": resume_batch_index,
#         "val_auc": val_auc,
#         "best_val_auc": best_val_auc,
#         "val_auc_history": val_auc_history,
#         "train_ids": train_ids,
#         "val_ids": val_ids,
#         "rng_states": get_rng_states(),
#         "config": CONFIG,
#         "dino_dim": DINO_DIM,
#         "architecture": "multi_head_v1",  # [MULTI-HEAD MOD] tag so old/new checkpoints are never confused
#         "saved_at": datetime.now(timezone.utc).isoformat(),
#     }
#     torch.save(ckpt, path)


# def load_checkpoint(path):
#     # [MULTI-HEAD MOD] weights_only=False: explicit for PyTorch 2.6+, this is a
#     # trusted, self-generated checkpoint (contains plain tensors + python config/lists).
#     ckpt = torch.load(path, map_location=device, weights_only=False)

#     if ckpt.get("architecture") != "multi_head_v1":
#         raise ValueError(
#             f"{path} does not look like a multi-head checkpoint "
#             f"(architecture={ckpt.get('architecture')!r}). Refusing to resume from an "
#             "old-architecture checkpoint into the new model -- train from scratch instead."
#         )

#     model.projection.load_state_dict(ckpt["projection_state_dict"])
#     model.window_mil.load_state_dict(ckpt["window_mil_state_dict"])
#     model.series_mil.load_state_dict(ckpt["series_mil_state_dict"])
#     model.head.load_state_dict(ckpt["head_state_dict"])
#     optimizer.load_state_dict(ckpt["optimizer_state_dict"])
#     scaler.load_state_dict(ckpt["scaler_state_dict"])
#     if "rng_states" in ckpt:
#         set_rng_states(ckpt["rng_states"])

#     # [MULTI-HEAD MOD] Explicit split verification before resuming, as requested.
#     if ckpt.get("train_ids") != train_ids or ckpt.get("val_ids") != val_ids:
#         print("WARNING: the train/val split stored in this checkpoint does not match the split "
#               "just computed in this session. Continuing with the freshly computed split, but "
#               "this should not happen if CONFIG/SAMPLE_CSV are unchanged.")
#     else:
#         print("Checkpoint split verified: matches the current train/val split exactly.")

#     return ckpt["epoch"], ckpt["resume_batch_index"], ckpt["val_auc"], ckpt["best_val_auc"], ckpt["val_auc_history"]


In [ ]:

def get_rng_states():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state()
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state



def set_rng_states(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])

    # Restore CPU RNG state as a CPU ByteTensor
    torch_state = torch.as_tensor(
        state["torch"], dtype=torch.uint8, device="cpu"
    )
    torch.set_rng_state(torch_state)

    # Restore CUDA RNG states, if available
    if torch.cuda.is_available() and "torch_cuda" in state:
        cuda_states = [
            torch.as_tensor(s, dtype=torch.uint8, device="cpu")
            for s in state["torch_cuda"]
        ]
        torch.cuda.set_rng_state_all(cuda_states)



def save_checkpoint(path, epoch, resume_batch_index,
                    val_auc, best_val_auc, val_auc_history):
    ckpt = {
        "task_state_dicts": [task.state_dict() for task in model.tasks],
        "optimizer_states": [opt.state_dict() for opt in optimizers],
        "scaler_states": [scaler.state_dict() for scaler in scalers],
        "epoch": epoch,
        "resume_batch_index": resume_batch_index,
        "val_auc": val_auc,
        "best_val_auc": best_val_auc,
        "val_auc_history": val_auc_history,
        "train_ids": train_ids,
        "val_ids": val_ids,
        "rng_states": get_rng_states(),
        "config": CONFIG,
        "dino_dim": DINO_DIM,
        "architecture": "independent_specialists_v1",
        # [GOLD-VAL MOD] Explicit tag: best_val_auc / val_auc in this checkpoint
        # are the 58-study gold macro ROC-AUC, not a random-split AUC.
        "validation_protocol": "gold_58_v1",
        "n_gold_val_studies": len(val_ids),
        "saved_at": datetime.now(timezone.utc).isoformat()
    }
    torch.save(ckpt, path)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device, weights_only=False)

    if ckpt.get("architecture") != "independent_specialists_v1":
        raise ValueError("Checkpoint architecture mismatch. Start a fresh run.")

    # [GOLD-VAL MOD] Refuse to resume a checkpoint saved under a different
    # validation protocol (e.g. the old random-80/20-split run) into this
    # gold-validation run -- best_val_auc would otherwise mean two different
    # things.
    if ckpt.get("validation_protocol") != "gold_58_v1":
        raise ValueError(
            f"Checkpoint validation_protocol mismatch (got {ckpt.get('validation_protocol')!r}, "
            "expected 'gold_58_v1'). This checkpoint was not produced by the gold-validation "
            "experiment -- start a fresh run instead of resuming from it."
        )

    if ckpt.get("train_ids") != train_ids or ckpt.get("val_ids") != val_ids:
        raise ValueError("Checkpoint train/validation split mismatch.")

    for task, state in zip(model.tasks, ckpt["task_state_dicts"]):
        task.load_state_dict(state)

    for opt, state in zip(optimizers, ckpt["optimizer_states"]):
        opt.load_state_dict(state)

    for scaler, state in zip(scalers, ckpt["scaler_states"]):
        scaler.load_state_dict(state)

    if "rng_states" in ckpt:
        set_rng_states(ckpt["rng_states"])

    return (
        ckpt["epoch"],
        ckpt["resume_batch_index"],
        ckpt["val_auc"],
        ckpt["best_val_auc"],
        ckpt["val_auc_history"]
    )


## 14. Resume logic

**[MULTI-HEAD MOD]**: `LATEST_CKPT_PATH` now points into the new, separate output
directory (Section 2), so this can only ever resume a multi-head run -- it will
never find (and therefore never accidentally resume from) one of the old
shared-head checkpoints. `load_checkpoint` additionally refuses to load a checkpoint
that isn't tagged `architecture == "multi_head_v1"`, as a second line of defense.


In [ ]:
start_epoch = 0
start_batch_index = 0
best_val_auc = -1.0
last_val_auc = 0.0
val_auc_history = []

if os.path.exists(LATEST_CKPT_PATH):
    start_epoch, start_batch_index, last_val_auc, best_val_auc, val_auc_history = load_checkpoint(LATEST_CKPT_PATH)
    print(
        f"Resumed from checkpoint: epoch={start_epoch}, batch_index={start_batch_index}, "
        f"best_val_auc={best_val_auc:.4f}, {len(val_auc_history)} prior epoch(s) recorded"
    )
else:
    print("No multi-head checkpoint found. Training this new architecture from scratch, as intended "
          "(the old shared-head checkpoints are in a different directory and are never read here).")


## 15. Validation (gold macro ROC-AUC, NaN-safe for single-class columns)

**[GOLD-VAL MOD]**: validation loss is still computed against the soft labels (informational only), but validation ROC-AUC -- the metric used for model selection and early stopping -- is now computed against the 58 official gold binary labels from `train.csv`, never `batch["targets"]`. See Section 6a.


In [ ]:
# @torch.no_grad()
# def run_validation(val_ids, batch_size):
#     model.eval()
#     all_probs, all_targets, all_masks = [], [], []

#     for start in range(0, len(val_ids), batch_size):
#         batch_ids = val_ids[start:start + batch_size]
#         batch, _ = build_batch(batch_ids, embeddings_index, DINO_DIM)

#         wf = batch["window_features"].to(device, non_blocking=True)
#         wm = batch["window_mask"].to(device, non_blocking=True)
#         sm = batch["series_mask"].to(device, non_blocking=True)

#         with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=USE_AMP):
#             logits = model(wf, wm, sm)

#         # Sigmoid applied only here, for evaluation -- never before the training loss.
#         all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
#         all_targets.append(batch["targets"].numpy())
#         all_masks.append(batch["label_masks"].numpy())

#     all_probs = np.concatenate(all_probs, axis=0)
#     all_targets = np.concatenate(all_targets, axis=0)
#     all_masks = np.concatenate(all_masks, axis=0)

#     per_class_auc = {}
#     for c, col_name in enumerate(label_cols):
#         valid = all_masks[:, c] > 0
#         y_true = all_targets[valid, c]
#         y_score = all_probs[valid, c]
#         if len(np.unique(y_true)) < 2:
#             per_class_auc[col_name] = float("nan")  # reported, excluded from the mean below
#             continue
#         try:
#             per_class_auc[col_name] = roc_auc_score(y_true, y_score)
#         except Exception:
#             per_class_auc[col_name] = float("nan")

#     auc_values = list(per_class_auc.values())
#     finite_aucs = [a for a in auc_values if not np.isnan(a)]
#     mean_auc = float(np.mean(finite_aucs)) if finite_aucs else 0.0

#     model.train()
#     return mean_auc, per_class_auc


In [ ]:
# [GOLD-VAL MOD] Rewritten. val_ids here are always the 58 official gold studies.
# VALIDATION LOSS is still computed against the SOFT labels from build_batch()
# (same objective as training -- useful for monitoring, informational only).
# VALIDATION ROC-AUC -- the metric used for model selection / early stopping --
# is computed against the OFFICIAL BINARY GOLD labels (get_gold_label), NEVER
# batch["targets"] and NEVER the soft-label mask (gold labels are already
# complete for all 58 studies, so no masking is needed for AUC).
@torch.no_grad()
def run_validation(val_ids, batch_size):
    was_training = model.training
    model.eval()

    all_probs = []
    gold_targets_list = []
    val_loss_sum = 0.0
    val_mask_sum = 0.0

    for start in range(0, len(val_ids), batch_size):
        batch_ids = val_ids[start:start + batch_size]
        batch, _ = build_batch(batch_ids, embeddings_index, DINO_DIM)

        wf = batch["window_features"].to(device, non_blocking=True)
        wm = batch["window_mask"].to(device, non_blocking=True)
        sm = batch["series_mask"].to(device, non_blocking=True)

        # Soft labels/mask -- used ONLY for the informational validation loss below.
        soft_targets = batch["targets"].to(device, non_blocking=True)
        soft_masks = batch["label_masks"].to(device, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE_TYPE,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            logits = model(wf, wm, sm)

        # VALIDATION LOSS -> soft labels (informational; same objective as training).
        loss_per_elem = F.binary_cross_entropy_with_logits(
            logits.float(), soft_targets.float(), reduction="none"
        )
        val_loss_sum += (loss_per_elem * soft_masks).sum().item()
        val_mask_sum += soft_masks.sum().item()

        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())

        # VALIDATION ROC-AUC -> official binary gold labels, NOT batch["targets"].
        gold_batch = np.stack([get_gold_label(sid) for sid in batch_ids])
        gold_targets_list.append(gold_batch)

    val_loss = val_loss_sum / max(val_mask_sum, 1.0)

    all_probs = np.concatenate(all_probs, axis=0)
    gold_targets = np.concatenate(gold_targets_list, axis=0)

    per_class_auc = {}
    for c, col_name in enumerate(label_cols):
        y_true = gold_targets[:, c]
        y_score = all_probs[:, c]

        if len(np.unique(y_true)) < 2:
            per_class_auc[col_name] = float("nan")
        else:
            per_class_auc[col_name] = roc_auc_score(y_true, y_score)

    finite_aucs = [
        a for a in per_class_auc.values() if np.isfinite(a)
    ]
    mean_auc = float(np.mean(finite_aucs)) if finite_aucs else float("nan")

    if was_training:
        model.train()

    return val_loss, mean_auc, per_class_auc


## 16. Training loop (UNCHANGED logic: optimizer, batching, AMP, grad clipping,
checkpoint cadence, and early-stopping patience are all identical to the original
notebook -- only the model being trained has changed)

This is the single training entry point. There is no other training call left
anywhere in this notebook (the old shared-head model's training code has been
replaced in place here, not duplicated) -- running this cell cannot accidentally
retrain the old architecture.


In [ ]:


def append_history_row(epoch, train_loss, val_loss, val_auc, lr):
    row = pd.DataFrame([{
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_mean_auc": val_auc,
        "learning_rate": lr,
        "timestamp": datetime.utcnow().isoformat(),
    }])

    file_exists = os.path.exists(HISTORY_CSV_PATH)

    row.to_csv(
        HISTORY_CSV_PATH,
        mode="a" if file_exists else "w",
        header=not file_exists,
        index=False
    )


def train():
    global start_epoch, start_batch_index, best_val_auc
    global last_val_auc, val_auc_history

    epochs_without_improvement = 0

    for epoch in range(start_epoch, CONFIG["NUM_EPOCHS"]):
        model.train()
        order = train_ids.copy()
        random.Random(CONFIG["SHUFFLE_SEED"] + epoch).shuffle(order)
        batches = [
            order[i:i + CONFIG["BATCH_SIZE"]]
            for i in range(0, len(order), CONFIG["BATCH_SIZE"])
        ]

        batch_start = start_batch_index if epoch == start_epoch else 0
        task_loss_sums = np.zeros(len(model.tasks), dtype=float)
        task_loss_counts = np.zeros(len(model.tasks), dtype=int)
        studies_since_ckpt = 0

        pbar = tqdm(range(batch_start, len(batches)), desc=f"Epoch {epoch}")

        for batch_idx in pbar:
            batch_ids = batches[batch_idx]
            batch, n_missing = build_batch(batch_ids, embeddings_index, DINO_DIM)

            wf = batch["window_features"].to(device, non_blocking=True)
            wm = batch["window_mask"].to(device, non_blocking=True)
            sm = batch["series_mask"].to(device, non_blocking=True)
            targets = batch["targets"].to(device, non_blocking=True)
            masks = batch["label_masks"].to(device, non_blocking=True)

            # Each abnormality has its own forward/backward/update.
            for task_idx, (task_model, optimizer, scaler) in enumerate(
                zip(model.tasks, optimizers, scalers)
            ):
                if masks[:, task_idx].sum().item() == 0:
                    continue

                optimizer.zero_grad(set_to_none=True)

                # with torch.autocast(
                #     device_type=DEVICE_TYPE,
                #     dtype=torch.float16,
                #     enabled=USE_AMP
                # ):
                #     logits = task_model(wf, wm, sm)
                #     loss = masked_task_bce_loss(
                #         logits, targets, masks, task_idx
                #     )
                

                # if loss is None or not torch.isfinite(loss):
                #     continue

                
                with torch.autocast(
                    device_type=DEVICE_TYPE,
                    dtype=torch.float16,
                    enabled=USE_AMP
                ):
                    logits = task_model(wf, wm, sm)
                    bce_loss = masked_task_bce_loss(
                        logits, targets, masks, task_idx
                    )
                
                if bce_loss is None or not torch.isfinite(bce_loss):
                    continue
                
                # L1 regularization on this task's classifier
                l1_penalty = classifier_l1_penalty(task_model)
                
                # Combined loss used for backpropagation
                loss = bce_loss + CONFIG["L1_LAMBDA"] * l1_penalty
                
                if not torch.isfinite(loss):
                    continue


                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    task_model.parameters(), CONFIG["MAX_GRAD_NORM"]
                )

                scaler.step(optimizer)
                scaler.update()

                task_loss_sums[task_idx] += bce_loss.detach().item()
                task_loss_counts[task_idx] += 1

            studies_since_ckpt += len(batch_ids)
            pbar.set_postfix(
                avg_task_loss=np.mean([
                    task_loss_sums[i] / max(task_loss_counts[i], 1)
                    for i in range(len(model.tasks))
                ])
            )

            if studies_since_ckpt >= CONFIG["CHECKPOINT_EVERY_N_STUDIES"]:
                save_checkpoint(
                    LATEST_CKPT_PATH, epoch, batch_idx + 1,
                    last_val_auc, best_val_auc, val_auc_history
                )
                studies_since_ckpt = 0

        per_task_train_loss = {
            name: (
                task_loss_sums[i] / task_loss_counts[i]
                if task_loss_counts[i] else float("nan")
            )
            for i, name in enumerate(label_cols)
        }

        mean_train_loss = np.nanmean(list(per_task_train_loss.values()))
        current_lr = CONFIG["LR"]

        val_loss, val_auc, per_class_auc = run_validation(
            val_ids, CONFIG["BATCH_SIZE"]
        )
        
        print(
            f"Epoch {epoch} | "
            f"train_loss={mean_train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_mean_auc={val_auc:.4f}"
        )
        last_val_auc = val_auc
        val_auc_history.append(val_auc)

        print(
            f"\nEpoch {epoch} | mean task train loss={mean_train_loss:.4f}"
            f" | val macro AUC={val_auc:.4f}"
        )
        for name in label_cols:
            print(
                f"  {name}: train_loss={per_task_train_loss[name]:.4f}"
                f" | val_auc={per_class_auc.get(name, float('nan'))}"
            )

        append_history_row(epoch, mean_train_loss, val_loss, val_auc, current_lr)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_without_improvement = 0
            save_checkpoint(
                BEST_CKPT_PATH, epoch + 1, 0,
                val_auc, best_val_auc, val_auc_history
            )
            print("Saved best checkpoint:", BEST_CKPT_PATH)
        else:
            epochs_without_improvement += 1

        save_checkpoint(
            LATEST_CKPT_PATH, epoch + 1, 0,
            val_auc, best_val_auc, val_auc_history
        )
        start_batch_index = 0

        if epochs_without_improvement >= CONFIG["EARLY_STOPPING_PATIENCE"]:
            print("Early stopping triggered.")
            break

    print("Training complete. Best validation macro AUC:", best_val_auc)


train()


## 17. Final verification: reload the best checkpoint into a fresh model instance


In [ ]:
# print("Loading best checkpoint and evaluating on the fixed validation set...")

# final_model = OrthoDinoMILModel(
#     DINO_DIM, CONFIG["PROJECTED_DIM"], CONFIG["ATTENTION_HIDDEN_DIM"], CONFIG["NUM_CLASSES"],
#     CONFIG["MULTIHEAD_HIDDEN_DIMS"], CONFIG["MULTIHEAD_DROPOUT"],
# ).to(device)

# # [MULTI-HEAD MOD] weights_only=False: explicit for PyTorch 2.6+, trusted self-generated file.
# best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
# final_model.projection.load_state_dict(best_ckpt["projection_state_dict"])
# final_model.window_mil.load_state_dict(best_ckpt["window_mil_state_dict"])
# final_model.series_mil.load_state_dict(best_ckpt["series_mil_state_dict"])
# final_model.head.load_state_dict(best_ckpt["head_state_dict"])
# final_model.eval()

# print(f"Loaded best checkpoint from epoch index {best_ckpt['epoch'] - 1} "
#       f"(stored best_val_auc={best_ckpt['best_val_auc']:.4f})")


In [ ]:

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)

# [ARCHITECTURE MOD] dropped the two trailing args -- OrthoDinoMILModel no
# longer takes head_hidden_dims/head_dropout (single Linear(512,1) classifier).
final_model = OrthoDinoMILModel(
    DINO_DIM,
    CONFIG["PROJECTED_DIM"],
    CONFIG["ATTENTION_HIDDEN_DIM"],
    CONFIG["NUM_CLASSES"],
).to(device)

for task, state in zip(final_model.tasks, best_ckpt["task_state_dicts"]):
    task.load_state_dict(state)

final_model.eval()
print("Loaded best independent-specialist checkpoint.")


In [ ]:
# plot: train vs validation

import pandas as pd
import matplotlib.pyplot as plt

class EmbeddingProjection(nn.Module):
    """Project frozen DINOv3 embeddings into a trainable feature space."""
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.norm(self.act(self.proj(x)))


def masked_mean_pool(x, mask):
    """
    x:    [B, ..., N, D]
    mask: [B, ..., N]
    Returns: [B, ..., D]
    """
    mask = mask.unsqueeze(-1).to(dtype=x.dtype)
    summed = (x * mask).sum(dim=-2)
    counts = mask.sum(dim=-2).clamp(min=1e-8)
    return summed / counts


class IndependentTaskModel(nn.Module):
    """One independent specialist for a single abnormality."""

    def __init__(self, dino_dim, projected_dim=512):
        super().__init__()

        # Independent feature projection for this task
        self.projection = EmbeddingProjection(
            dino_dim, projected_dim
        )

        # Task-specific neural network
        self.classifier = nn.Sequential(
            nn.Linear(projected_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),

            nn.Linear(128, 1)
        )

    def forward(self, window_features, window_mask, series_mask):
        # [B, S, W, D] -> [B, S, W, 512]
        x = self.projection(window_features)

        # Masked mean over windows -> [B, S, 512]
        series_features = masked_mean_pool(x, window_mask)

        # Exclude series with no valid windows
        valid_series = (
            (series_mask > 0) & (window_mask.sum(dim=-1) > 0)
        ).to(dtype=x.dtype)

        # Masked mean over series -> [B, 512]
        study_features = masked_mean_pool(
            series_features, valid_series
        )

        # [B, 512] -> [B, 1] -> [B]
        logits = self.classifier(study_features).squeeze(-1)
        return logits


class OrthoDinoMILModel(nn.Module):
    """12 independent specialists, one for each abnormality."""

    def __init__(
        self, dino_dim, projected_dim=512,
        attn_hidden=None, num_classes=12
    ):
        super().__init__()

        # attn_hidden retained for compatibility with existing call sites.
        self.tasks = nn.ModuleList([
            IndependentTaskModel(dino_dim, projected_dim)
            for _ in range(num_classes)
        ])

    def forward_task(
        self, task_idx, window_features, window_mask, series_mask
    ):
        return self.tasks[task_idx](
            window_features, window_mask, series_mask
        )

    def forward(self, window_features, window_mask, series_mask):
        # Each specialist outputs [B]; combined output is [B, 12]
        logits = [
            task(window_features, window_mask, series_mask)
            for task in self.tasks
        ]
        return torch.stack(logits, dim=1)


model = OrthoDinoMILModel(
    DINO_DIM,
    CONFIG["PROJECTED_DIM"],
    CONFIG["ATTENTION_HIDDEN_DIM"],
    CONFIG["NUM_CLASSES"]
).to(device)

print(model)
print("Independent specialists:", len(model.tasks))

history = pd.read_csv(HISTORY_CSV_PATH)
history = history.sort_values("epoch")

plt.figure(figsize=(9, 5))
plt.plot(history["epoch"], history["train_loss"], marker="o",
         label="Training Loss")
plt.plot(history["epoch"], history["val_loss"], marker="o",
         label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# [GOLD-VAL MOD] Rewritten to use the official gold labels for AUC, never
# batch["targets"] (which holds continuous soft labels and previously caused
# sklearn's "continuous format is not supported" error).
@torch.no_grad()
def evaluate_with_model(eval_model, val_ids, batch_size):
    all_probs = []
    gold_targets_list = []
    for start in range(0, len(val_ids), batch_size):
        batch_ids = val_ids[start:start + batch_size]
        batch, _ = build_batch(batch_ids, embeddings_index, DINO_DIM)
        wf = batch["window_features"].to(device)
        wm = batch["window_mask"].to(device)
        sm = batch["series_mask"].to(device)
        logits = eval_model(wf, wm, sm)
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        gold_targets_list.append(np.stack([get_gold_label(sid) for sid in batch_ids]))

    all_probs = np.concatenate(all_probs, axis=0)
    gold_targets = np.concatenate(gold_targets_list, axis=0)

    per_class_auc = {}
    for c, col_name in enumerate(label_cols):
        y_true = gold_targets[:, c]
        y_score = all_probs[:, c]
        if len(np.unique(y_true)) < 2:
            per_class_auc[col_name] = float("nan")
            continue
        try:
            per_class_auc[col_name] = roc_auc_score(y_true, y_score)
        except Exception:
            per_class_auc[col_name] = float("nan")

    finite_aucs = [a for a in per_class_auc.values() if not np.isnan(a)]
    mean_auc = float(np.mean(finite_aucs)) if finite_aucs else 0.0
    return mean_auc, per_class_auc


final_mean_auc, final_per_class_auc = evaluate_with_model(final_model, val_ids, CONFIG["BATCH_SIZE"])

print("=" * 70)
print("FINAL VALIDATION RESULTS (best multi-head checkpoint, freshly loaded model)")
print("=" * 70)
print(f"Mean validation ROC-AUC: {final_mean_auc:.4f}")
for col_name, auc in final_per_class_auc.items():
    print(f"  {col_name:30s} AUC: {auc if not np.isnan(auc) else 'NaN (single class)'}")


## 18. Save the complete final model and demonstrate inference on one study

In [ ]:
# torch.save({
#     "projection_state_dict": final_model.projection.state_dict(),
#     "window_mil_state_dict": final_model.window_mil.state_dict(),
#     "series_mil_state_dict": final_model.series_mil.state_dict(),
#     "head_state_dict": final_model.head.state_dict(),
#     "dino_dim": DINO_DIM,
#     "config": CONFIG,
#     "architecture": "multi_head_v1",
#     "final_val_mean_auc": final_mean_auc,
#     "final_per_class_auc": final_per_class_auc,
# }, FINAL_MODEL_PATH)
# print("Saved complete final model to:", FINAL_MODEL_PATH)


In [ ]:

torch.save({
    "task_state_dicts": [
        task.state_dict() for task in final_model.tasks
    ],
    "dino_dim": DINO_DIM,
    "config": CONFIG,
    "architecture": "independent_specialists_v1",
    "final_val_mean_auc": final_mean_auc,
    "final_per_class_auc": final_per_class_auc,
}, FINAL_MODEL_PATH)

print("Saved independent specialists to:", FINAL_MODEL_PATH)


In [ ]:
# # Demonstrate loading the saved model in a completely fresh instance
# demo_model = OrthoDinoMILModel(
#     DINO_DIM, CONFIG["PROJECTED_DIM"], CONFIG["ATTENTION_HIDDEN_DIM"], CONFIG["NUM_CLASSES"],
#     CONFIG["MULTIHEAD_HIDDEN_DIMS"], CONFIG["MULTIHEAD_DROPOUT"],
# ).to(device)

# # [MULTI-HEAD MOD] weights_only=False: explicit for PyTorch 2.6+, trusted self-generated file.
# demo_ckpt = torch.load(FINAL_MODEL_PATH, map_location=device, weights_only=False)
# demo_model.projection.load_state_dict(demo_ckpt["projection_state_dict"])
# demo_model.window_mil.load_state_dict(demo_ckpt["window_mil_state_dict"])
# demo_model.series_mil.load_state_dict(demo_ckpt["series_mil_state_dict"])
# demo_model.head.load_state_dict(demo_ckpt["head_state_dict"])
# demo_model.eval()

# demo_study_id = val_ids[0]
# demo_batch, _ = build_batch([demo_study_id], embeddings_index, DINO_DIM)

# with torch.no_grad():
#     demo_logits = demo_model(
#         demo_batch["window_features"].to(device),
#         demo_batch["window_mask"].to(device),
#         demo_batch["series_mask"].to(device),
#     )
#     demo_probs = torch.sigmoid(demo_logits).cpu().numpy()[0]  # sigmoid only here, for inference

# print(f"Inference demo on study {demo_study_id} using the freshly-loaded saved multi-head model:")
# for col_name, prob in zip(label_cols, demo_probs):
#     print(f"  {col_name:30s} p={prob:.4f}")


In [ ]:

# [ARCHITECTURE MOD] dropped the two trailing args -- OrthoDinoMILModel no
# longer takes head_hidden_dims/head_dropout (single Linear(512,1) classifier).
demo_model = OrthoDinoMILModel(
    DINO_DIM,
    CONFIG["PROJECTED_DIM"],
    CONFIG["ATTENTION_HIDDEN_DIM"],
    CONFIG["NUM_CLASSES"],
).to(device)

demo_ckpt = torch.load(FINAL_MODEL_PATH, map_location=device, weights_only=False)

for task, state in zip(demo_model.tasks, demo_ckpt["task_state_dicts"]):
    task.load_state_dict(state)

demo_model.eval()

demo_study_id = val_ids[0]
demo_batch, _ = build_batch([demo_study_id], embeddings_index, DINO_DIM)

with torch.no_grad():
    demo_logits = demo_model(
        demo_batch["window_features"].to(device),
        demo_batch["window_mask"].to(device),
        demo_batch["series_mask"].to(device)
    )
    demo_probs = torch.sigmoid(demo_logits).cpu().numpy()[0]

for name, prob in zip(label_cols, demo_probs):
    print(f"{name:30s} p={prob:.4f}")


## 19. Final summary, output file sizes, and comparison against the old shared-head model

**[MULTI-HEAD MOD]**: the block below is new. It only reports numbers that were
actually computed in this run (`final_mean_auc`, `final_per_class_auc`,
`val_auc_history`, `best_val_auc`) plus whatever the OLD shared-head notebook
already saved to `ORTHO_DINO_MIL_final_model.pt` in its own output directory, if
that file happens to exist. Nothing here is fabricated -- if the old file isn't
present (e.g. that notebook was never run in this session), the comparison is
skipped and says so explicitly.


In [ ]:
print("=" * 70)
print("ORTHOFOUNDATION DINO MULTI-HEAD -- RUN SUMMARY")
print("=" * 70)
print(f"Best validation mean ROC-AUC (this run): {best_val_auc:.4f}")
print(f"Final (reloaded) validation mean ROC-AUC: {final_mean_auc:.4f}")

print("\nValidation AUC history (by epoch):")
for i, auc in enumerate(val_auc_history):
    print(f"  Epoch {i}: {auc:.4f}")

# [MULTI-HEAD MOD] Best-effort comparison against the old shared-head model's saved
# results. Only prints real, previously-saved numbers -- never invented ones.
print("\n" + "=" * 70)
print("COMPARISON AGAINST OLD SHARED-HEAD MODEL (if available)")
print("=" * 70)
if os.path.exists(OLD_FINAL_MODEL_PATH):
    try:
        old_ckpt = torch.load(OLD_FINAL_MODEL_PATH, map_location="cpu", weights_only=False)
        old_mean_auc = old_ckpt.get("final_val_mean_auc")
        old_per_class_auc = old_ckpt.get("final_per_class_auc", {})

        if old_mean_auc is not None:
            print(f"Old shared-head mean ROC-AUC:  {old_mean_auc:.4f}")
            print(f"New multi-head mean ROC-AUC:   {final_mean_auc:.4f}")
            print(f"Difference (new - old):        {final_mean_auc - old_mean_auc:+.4f}")

            print("\nPer-class comparison:")
            for col_name in label_cols:
                old_v = old_per_class_auc.get(col_name, float("nan"))
                new_v = final_per_class_auc.get(col_name, float("nan"))
                old_str = f"{old_v:.4f}" if not (old_v is None or np.isnan(old_v)) else "NaN"
                new_str = f"{new_v:.4f}" if not np.isnan(new_v) else "NaN"
                print(f"  {col_name:30s} old={old_str:>8s}  new={new_str:>8s}")
        else:
            print(f"Found {OLD_FINAL_MODEL_PATH} but it has no 'final_val_mean_auc' field to compare against.")
    except Exception as e:
        print(f"Found {OLD_FINAL_MODEL_PATH} but could not load it for comparison: {e}")
else:
    print(f"No old shared-head final model found at {OLD_FINAL_MODEL_PATH} -- "
          "skipping comparison (this is expected if that notebook hasn't been run in this session).")

print("\nOutput files (new multi-head run):")
output_paths = [
    LATEST_CKPT_PATH,
    BEST_CKPT_PATH,
    FINAL_MODEL_PATH,
    HISTORY_CSV_PATH,
    CONFIG_JSON_PATH,
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_train_ids.csv"),
    os.path.join(CONFIG["OUTPUT_DIR"], "ORTHO_DINO_MULTIHEAD_val_ids.csv"),
]
for p in output_paths:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024**2)
        print(f"  {p}  ({size_mb:.2f} MB)")
    else:
        print(f"  {p}  -- MISSING")

print(
    "\nReminder: /kaggle/working is NOT permanently stored. Download "
    f"{CONFIG['OUTPUT_DIR']} now, or use Kaggle's 'Save Version -> Save & Run All' "
    "so these files are attached as a notebook Output before the session ends."
)
